# ResNet-50 y ablación desde cero (R4) — papa — Colab T4
Corre **ResNet-50 desde cero (semillas 0, 1, 2)**, **ViT-S/16 desde cero (semillas 1, 2)** y **ResNet-50 preentrenada (semillas 1, 2)**.
Ya hechas: ViT desde cero s0 (Colab) y ResNet preentrenada s0 (PC). En la GTX 1650 Ti local el fp16 colapsa la validación, por eso el resto va en la T4.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`.
Usa el mismo `colab_bundle.zip` que ya está en `MyDrive/MIA203/`. Los resultados se guardan en Drive y, si Colab se corta, al volver a correr se **saltan las corridas terminadas**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil
DRIVE = '/content/drive/MyDrive/MIA203'
bundle = f'{DRIVE}/colab_bundle.zip'
if not os.path.exists(bundle):
    from google.colab import files
    os.makedirs(DRIVE, exist_ok=True)
    print('No encontré colab_bundle.zip en Drive: súbelo ahora')
    up = files.upload()
    shutil.move(list(up)[0], bundle)

PROY = '/content/proyecto'
shutil.rmtree(PROY, ignore_errors=True)
zipfile.ZipFile(bundle).extractall(PROY)
os.chdir(PROY)
RES = f'{DRIVE}/results'
os.makedirs(f'{RES}/logs', exist_ok=True)
!pip -q install timm==1.0.30
import torch, timm, pandas as pd
print(torch.__version__, timm.__version__, torch.cuda.get_device_name(0))

# misma partición que la PC local
df = pd.read_csv('data/splits/papa_split.csv')
assert df.split.value_counts().to_dict() == {'train': 2112, 'test': 453, 'val': 450}
assert all(os.path.exists(p) for p in df.path)
print('partición OK')

## Entrenar (~15 min por corrida en T4, ~1 h 45 min en total)
Si Colab se desconecta, vuelve a ejecutar la celda de arriba y esta: retoma desde la siguiente corrida pendiente.

In [ ]:
cola = [('resnet50', 'false', 0), ('resnet50', 'false', 1), ('resnet50', 'false', 2),
        ('vit_small', 'false', 1), ('vit_small', 'false', 2),
        ('resnet50', 'true', 1), ('resnet50', 'true', 2)]
for cfg, pre, seed in cola:
    run = f"papa_{cfg}_{'pre' if pre == 'true' else 'scratch'}_s{seed}"
    if os.path.exists(f'{RES}/{run}/metrics.json'):
        print(f'{run}: ya existe, se salta'); continue
    print(f'===== {run} =====')
    !python src/train.py --config configs/{cfg}.yaml --seed {seed} --pretrained {pre} --set num_workers=2 out_dir={RES} 2>&1 | tee {RES}/logs/{run}.log

## Resumen

In [ ]:
import json, glob
filas = []
for f in sorted(glob.glob(f'{RES}/papa_*/metrics.json')):
    m = json.load(open(f))
    filas.append(dict(run=m['run'], best_epoch=m['best_epoch'], min=round(m['train_time_min'], 1),
                      val_f1=round(m['val']['macro_f1'], 4), **{f'test_{k}': round(v, 4) for k, v in m['test'].items()}))
t = pd.DataFrame(filas); display(t)

## Descargar
Solo las 7 corridas nuevas (con pesos `best.pt`) y sus logs. Descomprime el zip dentro de la carpeta `results/` del proyecto.

In [ ]:
runs = [f"papa_{c}_{'pre' if p == 'true' else 'scratch'}_s{s}" for c, p, s in cola]
with zipfile.ZipFile('/content/resultados_cero.zip', 'w') as z:
    for r in runs:
        for f in glob.glob(f'{RES}/{r}/*'):
            z.write(f, os.path.relpath(f, RES))
        if os.path.exists(f'{RES}/logs/{r}.log'):
            z.write(f'{RES}/logs/{r}.log', f'logs/colab/{r}.log')
print(len(runs), "corridas en el zip")
from google.colab import files
files.download('/content/resultados_cero.zip')